# AI-агент для бронирования билетов: Демонстрация

Демонстрация работы ИИ-агента службы поддержки. Архитектуры:
1. **Single-Agent ReAct**: Единый агент LangGraph.
2. **Multi-Agent System (MAS)**: Оркестратор с делегированием задач специализированным агентам и проверкой результатов Критиком.

### База данных
Используется in-memory база, инициализируемая из JSON файлов (`data/`):
- **Рейсы** (`flights.json`)
- **Бронирования** (`bookings.json`)
- **Правила** (`policies.json`)

## 1. Настройка окружения

In [1]:
import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv(dotenv_path='../.env')

from src.config import get_llm, REACT_SYSTEM_PROMPT
from src.database import DB, reset_db
from src.tools import ALL_TOOLS
from src.graph import build_react_graph, get_multi_agent_system

llm = get_llm(temperature=0.0)
print(f"Модель: {llm.model_name if hasattr(llm, 'model_name') else 'default'}")
print(f"В базе: рейсов — {len(DB['flights'])}, бронирований — {len(DB['bookings'])}")

c:\Users\alexander\.conda\envs\llm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Модель: default
В базе: рейсов — 8, бронирований — 2


c:\Users\alexander\.conda\envs\llm_env\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. Запуск единого ReAct агента

In [2]:
from src.config import extract_text
reset_db()

react_agent = build_react_graph(llm, REACT_SYSTEM_PROMPT, ALL_TOOLS)

query = "Покажи информацию по бронированию ABC123"
print(f"Пользователь: {query}\n")

state = react_agent.invoke({"messages": [("user", query)]})
print(f"Ответ агента:\n{extract_text(state['messages'][-1])}")

Пользователь: Покажи информацию по бронированию ABC123

[LOG] user: Покажи информацию по бронированию ABC123
[GUARD] is_on_topic -> yes
Ответ агента:
Информация по вашему бронированию ABC123:

*   **Пассажир:** Ivan Petrov
*   **Рейс:** SU2454
*   **Маршрут:** MOW (Москва) — PAR (Париж)
*   **Дата:** 2026-02-20
*   **Время вылета:** 08:30
*   **Класс обслуживания:** Эконом
*   **Статус:** Подтверждено
*   **Стоимость:** 25 000 руб.

Чем я могу вам помочь с этим бронированием?


## 3. Запуск мультиагентной системы (MAS) с Критиком

In [5]:
reset_db()

mas_coordinator = get_multi_agent_system(llm, max_revisions=1)

query = (
    "У меня бронирование BK-789 (Москва-Париж, эконом). Хочу перенести вылет на "
    "ближайший доступный рейс 16 апреля 2026. Сколько это будет стоить?"
)

print(f"Пользователь: {query}\n")
final_answer, specialist_results, critic_feedback = mas_coordinator.process_query_with_qc(query)

print("\n--- Оценка Критика ---")
print(f"Оценка: {critic_feedback.score}/10 | Одобрено: {critic_feedback.approved}")
print(f"Обоснование: {critic_feedback.reasoning}")

print("\n--- Итоговый ответ ---")
print(final_answer)

Пользователь: У меня бронирование BK-789 (Москва-Париж, эконом). Хочу перенести вылет на ближайший доступный рейс 16 апреля 2026. Сколько это будет стоить?

[GUARD] is_on_topic -> yes
[LOG] user: Get details for booking BK-789 (Moscow to Paris, economy).
[GUARD] is_on_topic -> yes
[LOG] user: Lookup policy for booking BK-789 changes (Moscow to Paris, economy).
[GUARD] is_on_topic -> yes
[LOG] user: Search for flights from Moscow to Paris on April 16, 2026, in economy class.
[GUARD] is_on_topic -> yes
[LOG] user: Calculate cost to change booking BK-789 to a flight on April 16, 2026.
[GUARD] is_on_topic -> yes

--- Оценка Критика ---
Оценка: 10.0/10 | Одобрено: True
Обоснование: The answer is complete, correct, and polite. It accurately reflects the specialist data provided and does not contain any risky or misleading recommendations. The answer also acknowledges the lack of specific policy information and suggests contacting the airline directly for more details.

--- Итоговый ответ ---